# Environment setup

In [1]:
!pip install torch torchvision wandb thop matplotlib

# Data & Custom Dataloader

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform_temp = transforms.Compose([transforms.ToTensor()])
train_dataset_temp = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_temp)
loader_temp = DataLoader(train_dataset_temp, batch_size=1000, shuffle=False, num_workers=0, pin_memory=True) # Set num_workers=0 for Windows

mean = torch.zeros(3)
std = torch.zeros(3)
total_images = 0
for images, _ in loader_temp:
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    total_images += batch_samples
mean /= total_images
std /= total_images

mean = mean.tolist()
std = std.tolist()
print("Calculated mean:", mean)
print("Calculated std:", std)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])


class TransformedSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

train_data = TransformedSubset(train_subset, transform=train_transform)
val_data = TransformedSubset(val_subset, transform=test_transform)

batch_size = 128
# Using num_workers=0 is crucial on Windows to avoid freezing/deadlocks in Jupyter Notebooks
num_workers = 0 
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

print("Train, validation, and test loaders are ready with normalized CIFAR-10 data.")

Using device: cuda
Files already downloaded and verified
Calculated mean: [0.49139973521232605, 0.48215848207473755, 0.4465309679508209]
Calculated std: [0.20230095088481903, 0.19941279292106628, 0.20096160471439362]
Files already downloaded and verified
Files already downloaded and verified
Train, validation, and test loaders are ready with normalized CIFAR-10 data.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(residual) 
        out = self.relu(out)
        return out

class CustomCNN(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.2):
        super(CustomCNN, self).__init__()
        
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=True)
        
        
        self.layer1 = ResidualBlock(32, 32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.drop1 = nn.Dropout2d(0.1)
        
        
        self.layer2 = ResidualBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.drop2 = nn.Dropout2d(0.1)
        
        
        self.layer3 = ResidualBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.drop3 = nn.Dropout2d(0.2)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.drop1(x)
        
        x = self.layer2(x)
        x = self.pool2(x)
        x = self.drop2(x)
        
        x = self.layer3(x)
        x = self.pool3(x)
        x = self.drop3(x)
        
        x = self.classifier(x)
        return x

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum = 0
    correct = 0
    total = 0
    # leave=False means this bar clears after completion (avoiding clutter)
    pbar = tqdm(loader, desc="Evaluating", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_sum += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss_sum/total, acc=correct/total)
    return loss_sum / total, correct / total


def train_one_epoch(model, loader, criterion, optimizer, scaler=None, epoch_idx=0):
    model.train()
    loss_sum = 0
    correct = 0
    total = 0
    # leave=True means this bar persists in the output logs
    pbar = tqdm(loader, desc=f"Epoch {epoch_idx+1} Train", leave=True)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        if scaler is not None:
            #
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        loss_sum += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss_sum/total, acc=correct/total)
    return loss_sum / total, correct / total

In [ ]:
def plot_grad_flow(named_parameters):
    '''Plots the gradients flowing through different layers in the net during training.
    Can be used for checking for possible gradient vanishing / exploding problems.
    
    Usage: Plug this function in Trainer class after loss.backward() as 
    "plot_grad_flow(self.model.named_parameters())" to visualize the gradient flow'''
    ave_grads = []
    max_grads = []
    layers = []
    for n, p in named_parameters:
        if(p.requires_grad) and ("bias" not in n):
            layers.append(n)
            if p.grad is not None:
                ave_grads.append(p.grad.abs().mean().cpu().item())
                max_grads.append(p.grad.abs().max().cpu().item())
            else:
                ave_grads.append(0)
                max_grads.append(0)
                
    fig = plt.figure(figsize=(10, 8))
    plt.bar(np.arange(len(max_grads)), max_grads, alpha=0.1, lw=1, color="c")
    plt.bar(np.arange(len(max_grads)), ave_grads, alpha=0.1, lw=1, color="b")
    plt.hlines(0, 0, len(ave_grads)+1, lw=2, color="k" )
    plt.xticks(range(0,len(ave_grads), 1), layers, rotation="vertical")
    plt.xlim(left=0, right=len(ave_grads))
    plt.ylim(bottom = -0.001, top=0.02) # zoom in on the lower gradient regions
    plt.xlabel("Layers")
    plt.ylabel("average gradient")
    plt.title("Gradient flow")
    plt.grid(True)
    plt.legend([plt.Line2D([0], [0], color="c", lw=4),
                plt.Line2D([0], [0], color="b", lw=4),
                plt.Line2D([0], [0], color="k", lw=4)], ['max-gradient', 'mean-gradient', 'zero-gradient'])
    return fig

def log_weight_histograms(model, epoch):
    for name, param in model.named_parameters():
        if "weight" in name:
            wandb.log({f"weights/{name}": wandb.Histogram(param.detach().cpu().numpy())}, step=epoch)
            if param.grad is not None:
                wandb.log({f"gradients/{name}": wandb.Histogram(param.grad.detach().cpu().numpy())}, step=epoch)

In [6]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\vasishth\_netrc.
wandb: Currently logged in as: m25csa007 (m25csa007-indian-institute-of-technology-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from thop import profile
import wandb

config = {
    "epochs": 30,
    "batch_size": 128,
    "lr": 0.001,
    "weight_decay": 1e-4,
    "architecture": "CustomCNN",
    "dataset": "CIFAR-10"
}

model = CustomCNN().to(device)
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

# Dummy forward pass for FLOPs
dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(model, inputs=(dummy_input,), verbose=False)
print(f"FLOPs: {flops:,.0f}, Params: {params:,.0f}")

wandb.init(
    project="Vasishth_Bhatt_M25CSA007_lab2_worksheet", 
    config=config)
wandb.log({"flops": flops, "params": params})

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

# Updated to torch.amp.GradScaler('cuda') to fix deprecation warning
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

best_val_acc = 0.0
patience = 10
patience_counter = 0


for epoch in range(config["epochs"]):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch_idx=epoch)
    
    # Visualizations (Gradient Flow & Weights)
    # Log gradient flow plot to WandB
    fig = plot_grad_flow(model.named_parameters())
    wandb.log({"gradient_flow": wandb.Image(fig)}, step=epoch+1)
    plt.close(fig)
    
    # Log histograms
    log_weight_histograms(model, epoch+1)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
    
    
    print(f" -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        print(f"    New best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load('best_model.pth', weights_only=True))
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"\nTest Loss={test_loss:.4f} Test Acc={test_acc:.4f}")
wandb.log({"test_loss": test_loss, "test_accuracy": test_acc})

FLOPs: 49,810,816, Params: 308,650


Epoch 1 Train: 100%|██████████| 313/313 [00:52<00:00,  5.91it/s, acc=0.213, loss=2.85]


 -> Val Loss: 1.9179 | Val Acc: 0.3598
    New best model saved!


Epoch 2 Train: 100%|██████████| 313/313 [00:52<00:00,  5.94it/s, acc=0.312, loss=1.97]


 -> Val Loss: 1.7293 | Val Acc: 0.4327
    New best model saved!


...

Epoch 49 Train: 100%|██████████| 313/313 [00:50<00:00,  6.20it/s, acc=0.839, loss=0.929]


 -> Val Loss: 0.8786 | Val Acc: 0.8604


Epoch 50 Train: 100%|██████████| 313/313 [00:52<00:00,  5.95it/s, acc=0.838, loss=0.929]


 -> Val Loss: 0.8791 | Val Acc: 0.8616
    New best model saved!



Test Loss=0.8993 Test Acc=0.8548


In [8]:
wandb.log({
    "best_val_accuracy": best_val_acc,
    "final_test_accuracy": test_acc
})
wandb.finish()

best_val_accuracy,▁
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
final_test_accuracy,▁
flops,▁
learning_rate,██████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
params,▁
test_accuracy,▁
test_loss,▁
train_accuracy,▁▂▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████████
train_loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...


# Findings and Observations

### Model Performance
- **Best Validation Accuracy**: [Log the best validation accuracy here]
- **Test Accuracy**: [Log the final test accuracy here]
- **FLOPs Count**: [Log the FLOPs count here]
- **Total Parameters**: [Log the total parameters here]

### Visualizations Analysis
- **Gradient Flow**:
  - [Describe if there were any vanishing or exploding gradients observed from the WandB plots]
  - [Comment on how the gradient magnitude changed over layers]

- **Weight Updates**:
  - [Observe the distribution of weights over epochs]
  - [Did the weights stabilize?]

### Conclusion
- [Summarize the overall performance of the Custom CNN on CIFAR-10]
- [Did the data augmentation help?]
- [Correlation between FLOPs/Params and performance]